# LSTMs for Text Classification

**Dataset:** AG_NEWS (News topic classification: World, Sports, Business, Sci/Tech)

**Instructions:** Complete the simple `# TODO` sections marked with `None`. Run all cells top-to-bottom to train and evaluate your model.

In [1]:
# Run this cell to install dependencies and load the dataset
!pip install -q datasets

In [2]:
import re
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import load_dataset

# Use GPU if available# Use GPU if available for faster training.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


### Step 1: Vocabulary & Preprocessing
Neural networks need numbers, not raw text. We will build a vocabulary to map words to integers.

In [3]:
def tokenizer(text):
    """Convert text to a list of lowercase alphanumeric tokens."""
    return re.findall(r"[a-z0-9]+", text.lower())

# Load the AG_NEWS dataset
ag_news = load_dataset('fancyzhx/ag_news')
train_data = ag_news['train']
test_data  = ag_news['test']

def yield_tokens(data_iter):
    """Generator that yields tokenized texts from the dataset."""
    for example in data_iter:
        yield tokenizer(example['text'])

from collections import Counter

# Count word frequencies in the training set
counter = Counter()
for tokens in yield_tokens(train_data):
    counter.update(tokens)

# Build vocabulary: <unk> and <pad> are added first
itos = ['<unk>', '<pad>'] + list(counter.keys())
stoi = {word: idx for idx, word in enumerate(itos)}
UNK_IDX = stoi['<unk>']
PAD_IDX = stoi['<pad>']

def numericalize(text):
    """Convert a text string into a list of integer token IDs."""
    return [stoi.get(tok, UNK_IDX) for tok in tokenizer(text)]

print(f"Vocabulary size: {len(itos):,}")

def collate_batch(batch):
    """
    Collate function for DataLoader.
    - Converts texts to padded tensors of token IDs.
    - Converts labels to a tensor.
    """
    label_list, text_list = [], []
    for example in batch:
        label_list.append(example['label'])  # labels are 0-3
        processed_text = torch.tensor(numericalize(example['text']), dtype=torch.int64)
        text_list.append(processed_text)

    # -----------------------------------------------------------------
    # Pad sequences so that all sentences in the batch have the same length.
    # batch_first=True gives shape (batch_size, max_seq_len)
    # padding_value=PAD_IDX fills extra positions with the <pad> token ID.
    # -----------------------------------------------------------------
    padded_texts = nn.utils.rnn.pad_sequence(
        text_list, batch_first=True, padding_value=PAD_IDX
    )
    labels = torch.tensor(label_list, dtype=torch.int64)

    return padded_texts, labels

# Use a small subset for fast training (optional, remove for full dataset)
train_list = list(train_data)[:5000]
test_list  = list(test_data)[:1000]

# Create DataLoaders
tr_ld = DataLoader(train_list, batch_size=32, shuffle=True, collate_fn=collate_batch)
te_ld = DataLoader(test_list, batch_size=32, shuffle=False, collate_fn=collate_batch)

README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.23MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Vocabulary size: 65,017


### Step 2: Build the LSTM Model
Construct a simple LSTM text classifier.

In [4]:
class SimpleLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        # Embedding layer: maps token IDs to dense vectors.
        # padding_idx ensures that <pad> tokens always get a zero vector.
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)

        # -----------------------------------------------------------------
        # LSTM layer: processes sequences and returns both outputs and hidden states.
        # batch_first=True makes input/output shape (batch, seq_len, feature).
        # -----------------------------------------------------------------
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)

        # -----------------------------------------------------------------
        # Fully connected layer: maps the final hidden state to class scores.
        # -----------------------------------------------------------------
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, text):
        # text shape: (batch_size, seq_len)
        embedded = self.embedding(text)          # -> (batch, seq_len, embed_dim)

        # -----------------------------------------------------------------
        # Pass through LSTM.
        # output: (batch, seq_len, hidden_dim) – all hidden states.
        # (hidden, cell): tuple of (num_layers, batch, hidden_dim)
        # -----------------------------------------------------------------
        output, (hidden, cell) = self.lstm(embedded)

        # For classification we use the final hidden state of the last LSTM layer.
        # hidden[-1] shape: (batch, hidden_dim)
        final_hidden = hidden[-1]

        # Apply the linear layer to get logits.
        logits = self.fc(final_hidden)           # -> (batch, num_classes)
        return logits

# Instantiate the model and move it to the appropriate device.
model = SimpleLSTM(vocab_size=len(itos), embed_dim=64, hidden_dim=128, num_classes=4).to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

Model parameters: 4,260,932


### Step 3: Train and Evaluate
Write the core steps of the PyTorch training loop.

In [5]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
criterion = nn.CrossEntropyLoss()

for epoch in range(3):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for texts, labels in tr_ld:
        texts, labels = texts.to(device), labels.to(device)

        # Reset gradients from previous iteration.
        optimizer.zero_grad()

        # Forward pass: compute predictions (logits).
        predictions = model(texts)

        # Compute the loss between predictions and true labels.
        loss = criterion(predictions, labels)

        # Backward pass: compute gradients.
        loss.backward()

        # Update model weights.
        optimizer.step()

        total_loss += loss.item()
        correct += (predictions.argmax(1) == labels).sum().item()
        total += labels.size(0)

    train_acc = correct / total

    # Evaluation phase
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for texts, labels in te_ld:
            texts, labels = texts.to(device), labels.to(device)
            preds = model(texts)
            val_correct += (preds.argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total
    print(f"Epoch {epoch+1}/3 | Train Acc: {train_acc:.1%} | Val Acc: {val_acc:.1%}")

Epoch 1/3 | Train Acc: 29.8% | Val Acc: 25.4%
Epoch 2/3 | Train Acc: 30.3% | Val Acc: 25.5%
Epoch 3/3 | Train Acc: 30.8% | Val Acc: 26.9%


### Step 4: Reflection
1. What does the `padding_idx` argument do in the `nn.Embedding` layer?
2. Why do we extract `hidden[-1]` instead of using the raw `output` from the LSTM for classification?

### Step 5: Inference
Now that the model is trained, let's use it to classify a brand new headline that it has never seen before.

In [8]:
class_names = ['World', 'Sports', 'Business', 'Sci/Tech']

def predict(text, model):
    """
    Classify a single text string using the trained model.
    Returns the predicted class name.
    """
    model.eval()

    # Convert raw text into a tensor of token IDs.
    text_tensor = torch.tensor(numericalize(text), dtype=torch.int64).to(device)

    # Add a batch dimension (batch size = 1).
    text_tensor = text_tensor.unsqueeze(0)   # shape: (1, seq_len)

    with torch.no_grad():
        # Forward pass to get logits.
        logits = model(text_tensor)

        # Get the index of the highest logit (the predicted class).
        predicted_idx = logits.argmax(1).item()

    return class_names[predicted_idx]

# Test on a few sample headlines
sample_headlines = [
    "Manchester United wins dramatic final in extra time",
    "Central bank raises interest rates to combat inflation",
    "NASA's new telescope captures images of distant galaxy",
    "Peace talks resume between the two neighboring countries"
]

for headline in sample_headlines:
    prediction = predict(headline, model)
    print(f"'{headline}' -> {prediction}")

'Manchester United wins dramatic final in extra time' -> Sci/Tech
'Central bank raises interest rates to combat inflation' -> Sci/Tech
'NASA's new telescope captures images of distant galaxy' -> Sci/Tech
'Peace talks resume between the two neighboring countries' -> Sci/Tech


In [10]:
# PREDICTIONS ON CUSTOM HEADLINES
import pandas as pd

sample_headlines = [
    # ---- Sports (5 headlines) ----
    "Manchester United wins dramatic final in extra time",
    "Serena Williams advances to Wimbledon semi-finals with straight-set victory",
    "Lakers secure playoff spot after overtime thriller against Warriors",
    "New Zealand claims Rugby World Cup in nail‑biting finish",
    "Olympic swimmer breaks world record by 0.2 seconds",

    # ---- Business & Finance (5 headlines) ----
    "Central bank raises interest rates to combat inflation",
    "Tech giant announces record quarterly profits, shares surge",
    "Oil prices drop sharply as OPEC increases production",
    "Startup raises $50 million in Series B funding round",
    "Global trade talks stall over tariff disputes",

    # ---- Sci/Tech (5 headlines) ----
    "NASA's new telescope captures images of distant galaxy",
    "Scientists develop vaccine that could end malaria epidemic",
    "Quantum computing breakthrough paves way for new materials",
    "AI model surpasses human performance in medical diagnosis",
    "SpaceX successfully lands rocket on floating platform",

    # ---- World News (5 headlines) ----
    "Peace talks resume between the two neighboring countries",
    "UN passes resolution to provide humanitarian aid to conflict zone",
    "Earthquake strikes major city, rescue efforts underway",
    "Election results signal shift in political landscape",
    "Diplomatic relations restored after decades of tension",

    # ---- Mixed / General (5 headlines) ----
    "New study shows that regular exercise improves mental health",
    "Celebrity chef opens new restaurant in downtown area",
    "Weather forecast predicts heavy snowfall for the weekend",
    "Local community rallies to save historic landmark",
    "Book by young author becomes instant bestseller",
]

# -----------------------------------------------------------------------------
# 2. Assign "actual" labels for each headline.
#    These are our best guesses for the true category.
# -----------------------------------------------------------------------------
actual_labels = [
    "Sports", "Sports", "Sports", "Sports", "Sports",
    "Business", "Business", "Business", "Business", "Business",
    "Sci/Tech", "Sci/Tech", "Sci/Tech", "Sci/Tech", "Sci/Tech",
    "World", "World", "World", "World", "World",
    "Sci/Tech",   # "New study shows that regular exercise improves mental health"
    "Business",   # "Celebrity chef opens new restaurant in downtown area"
    "World",      # "Weather forecast predicts heavy snowfall for the weekend"
    "World",      # "Local community rallies to save historic landmark"
    "Business",   # "Book by young author becomes instant bestseller"
]

# -----------------------------------------------------------------------------
# 3. Run predictions using the trained model.
# -----------------------------------------------------------------------------
predicted_labels = []
for headline in sample_headlines:
    pred = predict(headline, model)
    predicted_labels.append(pred)

# -----------------------------------------------------------------------------
# 4. Build a pandas DataFrame.
# -----------------------------------------------------------------------------
df_results = pd.DataFrame({
    "Sentence": sample_headlines,
    "Actual": actual_labels,
    "Predicted": predicted_labels
})

# -----------------------------------------------------------------------------
# 5. Display the full table.
# -----------------------------------------------------------------------------
print("\n" + "=" * 90)
print("CUSTOM HEADLINE CLASSIFICATION RESULTS")
print("=" * 90)
print(df_results.to_string(index=False))

# -----------------------------------------------------------------------------
# 6. Compute and show accuracy on this custom set.
#    This is an approximate metric because "Actual" is our own assignment.
# -----------------------------------------------------------------------------
accuracy = (df_results['Actual'] == df_results['Predicted']).mean()
print(f"\n✅ Accuracy on custom headlines: {accuracy:.1%}")

# -----------------------------------------------------------------------------
# 7. (Optional) Show where the model got it wrong.
# -----------------------------------------------------------------------------
errors = df_results[df_results['Actual'] != df_results['Predicted']]
if not errors.empty:
    print("\n--- Misclassified Headlines ---")
    print(errors.to_string(index=False))
else:
    print("\n🎉 All headlines were classified correctly!")


CUSTOM HEADLINE CLASSIFICATION RESULTS
                                                                   Sentence   Actual Predicted
                        Manchester United wins dramatic final in extra time   Sports  Sci/Tech
Serena Williams advances to Wimbledon semi-finals with straight-set victory   Sports  Sci/Tech
        Lakers secure playoff spot after overtime thriller against Warriors   Sports  Sci/Tech
                   New Zealand claims Rugby World Cup in nail‑biting finish   Sports  Business
                         Olympic swimmer breaks world record by 0.2 seconds   Sports     World
                     Central bank raises interest rates to combat inflation Business  Sci/Tech
                Tech giant announces record quarterly profits, shares surge Business  Business
                       Oil prices drop sharply as OPEC increases production Business     World
                       Startup raises $50 million in Series B funding round Business  Business
          